# Pseudo-4D Motion Recovery

Standard 3D MRI acquisitions are not designed to capture motion — but motion leaves a trace in k-space anyway.

The central column of k-space (the DC component) is acquired repeatedly throughout a scan. Its magnitude fluctuates with bulk motion — breathing, cardiac pulsation, patient movement. This signal can be extracted and used as a **motion surrogate** to sort k-space frames into motion phases, producing a pseudo-4D volume from a standard 3D acquisition.

This is the same principle used in **retrospective respiratory gating** in cardiac MRI.

> Note: The knee single-coil test set contains static scans. This notebook demonstrates the technique on simulated motion. The full method applies to dynamic acquisitions (cardiac, abdominal).

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from kode.io import load_kspace, root_sum_of_squares
from kode.pseudo4d import extract_motion_surrogate, sort_by_motion_phase

In [ ]:
# Simulate a time series by stacking slices as pseudo-time frames
# Each slice becomes one "time point" — demonstrates the surrogate extraction
H5_FILE = '../data/your_file.h5'

slices = []
for i in range(20):
    ks = load_kspace(H5_FILE, slice_idx=i)
    slices.append(ks)

# Stack: shape (time=20, coils, height, width)
kspace_timeseries = np.stack(slices, axis=0)
print(f'Time series shape: {kspace_timeseries.shape}')

In [ ]:
# Extract the motion surrogate signal
surrogate = extract_motion_surrogate(kspace_timeseries)

plt.figure(figsize=(10, 3))
plt.plot(surrogate, marker='o', color='steelblue', linewidth=1.5)
plt.xlabel('Frame index')
plt.ylabel('DC component magnitude')
plt.title('Motion Surrogate Signal — Extracted from K-Space DC Component')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../results/pseudo4d_surrogate.png', dpi=150)
plt.show()

In [ ]:
# Sort frames into 4 motion phases and reconstruct each
phases = sort_by_motion_phase(kspace_timeseries, n_phases=4)

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
phase_labels = ['Phase 1\n(min motion)', 'Phase 2', 'Phase 3', 'Phase 4\n(max motion)']

for ax, phase_kspace, label in zip(axes, phases, phase_labels):
    image = root_sum_of_squares(phase_kspace)
    ax.imshow(image, cmap='gray')
    ax.set_title(label, fontsize=10)
    ax.axis('off')

plt.suptitle('Pseudo-4D — Motion Phases Recovered from K-Space Time Structure', fontsize=13)
plt.tight_layout()
plt.savefig('../results/pseudo4d_phases.png', dpi=150, bbox_inches='tight')
plt.show()